# Step 4: Perform scGLUE integration on subsetted ATAC (120k cells total) and combined me and ac Cut&Tag

## Prepare data


In [1]:
import pybedtools
import scglue

/home/artem/miniconda3/envs/singlecell2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/artem/miniconda3/envs/singlecell2/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [2]:
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import sys

sys.path.append('./toolkit-project')
import src

In [3]:
cttag=ad.read_h5ad('../h5ad/emb8_14_comb_filter_250215.h5ad')

## Combine H3K27Ac and H3K27me3 in a single matrix before the integration

In [4]:
# the data features shouldn't be very stringently filtered. Maybe 25-26k features is good.
import scipy.sparse as sps

#stack features
combined = ad.AnnData(
    sps.hstack((
    cttag.layers['meth'],
    cttag.layers['acet']
    ))
)
combined.obs = cttag.obs.copy()
combined.obsm['X_multi_spectral'] = cttag.obsm['X_multi_spectral']


In [5]:
combined.var['chr'] = np.tile(
    cttag.var['chr'],
    2
)
combined.var['start'] = np.tile(
    cttag.var['start'],
    2
)
combined.var['end'] = np.tile(
    cttag.var['end'],
    2
)
combined.var['feature_type'] = np.hstack((
    np.array(['methylation'] * cttag.shape[1]),
    np.array(['acetylation'] * cttag.shape[1])
))

## Read in ATAC data

In [6]:
atac = ad.read_h5ad('../h5ad/pp_atac120k.h5ad')

In [7]:
atac.X = atac.layers['atac'].copy()
# data needs to be in .X for scglue
del atac.layers['atac']

In [8]:
atac.var['chrom'] = atac.var['chrom'].apply(lambda x: f'chr{x}')

In [9]:
def create_bed_df(s, expand: int = 0):
    df = pd.DataFrame(s, columns=["chrom","chromStart","chromEnd","name"])

    for v in ["score","strand","thickStart","thickEnd","itemRgb","blockCount","blockSizes","blockStarts"]:
        df[v] = "."
        
    df['nindex'] = df['name'].copy()
    df = df.set_index('nindex', drop=True)
    df.index.name = None
    
    if (expand != 0):
        df['chromStart'] = np.clip(df['chromStart'] - expand, 0, None)
        df['chromEnd'] += expand
        
    return scglue.genomics.Bed(df)

In [10]:
combined.var['id'] = combined.var['chr'] + combined.var['start'].astype('str') + combined.var['end'].astype('str') + combined.var['feature_type']
#combined.var

In [11]:
atac.var['id'] = atac.var.index.to_numpy().copy()

In [12]:
atac_bed = create_bed_df(atac.var[['chrom', 'chromStart', 'chromEnd', 'id']].to_numpy())

cttag_bed = create_bed_df(combined.var[['chr', 'start', 'end', 'id']].to_numpy())

# Make graph

In [13]:
cttag_bed

,chrom,chromStart,chromEnd,name,score,strand,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts
chr2L1000015000methylation,chr2L,10000,15000,chr2L1000015000methylation,.,.,.,.,.,.,.,.
chr2L100000105000methylation,chr2L,100000,105000,chr2L100000105000methylation,.,.,.,.,.,.,.,.
chr2L1000000010005000methylation,chr2L,10000000,10005000,chr2L1000000010005000methylation,.,.,.,.,.,.,.,.
chr2L1000500010010000methylation,chr2L,10005000,10010000,chr2L1000500010010000methylation,.,.,.,.,.,.,.,.
chr2L1001000010015000methylation,chr2L,10010000,10015000,chr2L1001000010015000methylation,.,.,.,.,.,.,.,.
...,...,...,...,...,...,...,...,...,...,...,...,...
chrX99750009980000acetylation,chrX,9975000,9980000,chrX99750009980000acetylation,.,.,.,.,.,.,.,.
chrX99800009985000acetylation,chrX,9980000,9985000,chrX99800009985000acetylation,.,.,.,.,.,.,.,.
chrX99850009990000acetylation,chrX,9985000,9990000,chrX99850009990000acetylation,.,.,.,.,.,.,.,.
chrX99900009995000acetylation,chrX,9990000,9995000,chrX99900009995000acetylation,.,.,.,.,.,.,.,.


In [16]:
atac_bed

,chrom,chromStart,chromEnd,name,score,strand,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts
chr2L:0-5000,chr2L,0,5000,chr2L:0-5000,.,.,.,.,.,.,.,.
chr2L:5000-10000,chr2L,5000,10000,chr2L:5000-10000,.,.,.,.,.,.,.,.
chr2L:10000-15000,chr2L,10000,15000,chr2L:10000-15000,.,.,.,.,.,.,.,.
chr2L:15000-20000,chr2L,15000,20000,chr2L:15000-20000,.,.,.,.,.,.,.,.
chr2L:20000-25000,chr2L,20000,25000,chr2L:20000-25000,.,.,.,.,.,.,.,.
...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:23520000-23525000,chrX,23520000,23525000,chrX:23520000-23525000,.,.,.,.,.,.,.,.
chrX:23525000-23530000,chrX,23525000,23530000,chrX:23525000-23530000,.,.,.,.,.,.,.,.
chrX:23530000-23535000,chrX,23530000,23535000,chrX:23530000-23535000,.,.,.,.,.,.,.,.
chrX:23535000-23540000,chrX,23535000,23540000,chrX:23535000-23540000,.,.,.,.,.,.,.,.


In [13]:
import networkx as nx

# overlap of bins graph
overlap_graph = scglue.genomics.window_graph(
    atac_bed, cttag_bed, 0,
    attr_fn=lambda l, r, d: {
        "weight": 1.0,
        "type": "overlap"
    }
)

overlap_graph = nx.DiGraph(overlap_graph)

(overlap_graph).number_of_edges()

window_graph: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 26393/26393 [00:00<00:00, 27993.73it/s]


46182

In [14]:
combined.var['connected'] = [overlap_graph.has_node(v) for v in combined.var['id']]
atac.var['connected'] = [overlap_graph.has_node(v) for v in atac.var['id']]
#atac2 = atac[:,atac.var['connected']]
#rna = rna[:,rna.var['connected']]
atac2 = atac
atac2.var_names = atac.var['id']

In [15]:
combined2 = combined
combined2.var_names = combined.var['id']

In [16]:
import itertools 

guidance = scglue.graph.compose_multigraph(overlap_graph, overlap_graph.reverse())

allids = list(itertools.chain(atac2.var['id'].to_list(), combined2.var['id'].to_list()))
# Assert all ids are there only once
assert np.unique(np.unique(allids,return_counts=True)[1]).shape[0] == 1

guidance = guidance.subgraph(nodes = allids).copy()

for item in allids:
    guidance.add_edge(item, item, weight=1.0, type="self-loop")
    
nx.set_edge_attributes(guidance, 1, "sign")

guidance.number_of_nodes()

72575

In [17]:
scglue.graph.check_graph(guidance, [combined2, atac2])

[INFO] check_graph: Checking variable coverage...
[INFO] check_graph: Checking edge attributes...
[INFO] check_graph: Checking self-loops...
[INFO] check_graph: Checking graph symmetry...
[INFO] check_graph: All checks passed!


In [18]:
nx.write_graphml(guidance, "../meta/atac_ct_guidance_250217.graphml.gz")

In [19]:
#  if the datatype isn't correct, use atac.X = atac.X.astype(np.uint32)
assert atac2.X.dtype == np.uint32
assert combined2.X.dtype == np.uint32

In [20]:
combined2.write_h5ad('../h5ad/pb-combined_250217.h5ad')
atac2.write_h5ad('../h5ad/pb-atac_250217.h5ad')

# DO the integration

In [25]:
import scglue
import anndata as ad
import networkx as nx
import numpy as np

In [21]:
combined = ad.read_h5ad('../h5ad/pb-combined_250217.h5ad')
atac = ad.read_h5ad('../h5ad/pb-atac_250217.h5ad')
guidance = nx.read_graphml("../meta/atac_ct_guidance_250217.graphml.gz")

In [22]:
# use_rep is very important parameter. You have to
# pick a good low dimensional embedding (but not 2D UMAP)

# for sccuttag use X_multi_spectral
# for atac use X_spectral

scglue.models.configure_dataset(
    combined, "NB", use_highly_variable=False,
    use_rep="X_multi_spectral"#, use_obs_names=True
)
scglue.models.configure_dataset(
    atac, "NB", use_highly_variable=False,
    use_rep="X_spectral"#, use_obs_names=True
)

In [23]:
#check if all features were successfully included in the guidance graph

assert np.array(combined.uns['__scglue__']['features']).shape[0] == combined.shape[1]
assert np.array(atac.uns['__scglue__']['features']).shape[0] == atac.shape[1]

In [24]:
glue = scglue.models.fit_SCGLUE(
    {"cttag": combined, "atac": atac}, guidance,
    #model=scglue.models.PairedSCGLUEModel,
    fit_kws={"directory": "glue"},
    init_kws={'latent_dim': 20}
)
glue.save("glue_cat2reps_atac_120_250217.dill")

[INFO] fit_SCGLUE: Pretraining SCGLUE model...
[INFO] autodevice: Using GPU 0 as computation device.


<frozen abc>:119: FutureWarning: SparseDataset is deprecated and will be removed in late 2024. It has been replaced by the public classes CSRDataset and CSCDataset.

For instance checks, use `isinstance(X, (anndata.experimental.CSRDataset, anndata.experimental.CSCDataset))` instead.

For creation, use `anndata.experimental.sparse_dataset(X)` instead.



[INFO] check_graph: Checking variable coverage...
[INFO] check_graph: Checking edge attributes...
[INFO] check_graph: Checking self-loops...
[INFO] check_graph: Checking graph symmetry...
[INFO] check_graph: All checks passed!
[INFO] SCGLUEModel: Setting `graph_batch_size` = 56698
[INFO] SCGLUEModel: Setting `max_epochs` = 48
[INFO] SCGLUEModel: Setting `patience` = 4
[INFO] SCGLUEModel: Setting `reduce_lr_patience` = 2
[INFO] SCGLUETrainer: Using training directory: "glue/pretrain"


/home/artem/miniconda3/envs/singlecell2/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
<frozen abc>:119: FutureWarning: SparseDataset is deprecated and will be removed in late 2024. It has been replaced by the public classes CSRDataset and CSCDataset.

For instance checks, use `isinstance(X, (anndata.experimental.CSRDataset, anndata.experimental.CSCDataset))` instead.

For creation, use `anndata.experimental.sparse_dataset(X)` instead.



[INFO] LRScheduler: Learning rate reduction: step 1
[INFO] SCGLUETrainer: [Epoch 10] train={'g_nll': 0.408, 'g_kl': 0.001, 'g_elbo': 0.409, 'x_cttag_nll': 0.064, 'x_cttag_kl': 0.0, 'x_cttag_elbo': 0.064, 'x_atac_nll': 0.411, 'x_atac_kl': 0.001, 'x_atac_elbo': 0.412, 'dsc_loss': 0.693, 'vae_loss': 0.493, 'gen_loss': 0.458}, val={'g_nll': 0.408, 'g_kl': 0.001, 'g_elbo': 0.409, 'x_cttag_nll': 0.065, 'x_cttag_kl': 0.0, 'x_cttag_elbo': 0.065, 'x_atac_nll': 0.41, 'x_atac_kl': 0.001, 'x_atac_elbo': 0.412, 'dsc_loss': 0.692, 'vae_loss': 0.493, 'gen_loss': 0.459}, 52.8s elapsed
[INFO] LRScheduler: Learning rate reduction: step 2
[INFO] LRScheduler: Learning rate reduction: step 3


...

/home/artem/miniconda3/envs/singlecell2/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


[INFO] SCGLUETrainer: [Epoch 10] train={'g_nll': 0.408, 'g_kl': 0.001, 'g_elbo': 0.409, 'x_cttag_nll': 0.064, 'x_cttag_kl': 0.0, 'x_cttag_elbo': 0.064, 'x_atac_nll': 0.411, 'x_atac_kl': 0.001, 'x_atac_elbo': 0.412, 'dsc_loss': 0.684, 'vae_loss': 0.493, 'gen_loss': 0.459}, val={'g_nll': 0.407, 'g_kl': 0.001, 'g_elbo': 0.409, 'x_cttag_nll': 0.064, 'x_cttag_kl': 0.0, 'x_cttag_elbo': 0.064, 'x_atac_nll': 0.413, 'x_atac_kl': 0.001, 'x_atac_elbo': 0.414, 'dsc_loss': 0.728, 'vae_loss': 0.495, 'gen_loss': 0.459}, 37.9s elapsed
[INFO] LRScheduler: Learning rate reduction: step 1
[INFO] LRScheduler: Learning rate reduction: step 2


2025-02-17 16:34:40,744 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


[INFO] EarlyStopping: Restoring checkpoint "16"...
[INFO] EarlyStopping: Restoring checkpoint "16"...


/home/artem/miniconda3/envs/singlecell2/lib/python3.12/site-packages/scglue/models/plugins.py:145: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded = torch.load(director

# Analyze and save

In [1]:
import scglue
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

/faststorage/project/PAN_illumina/people/artem/singlecell/jnb/venv/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [3]:
guidance = nx.read_graphml("../meta/atac_ct_guidance.graphml.gz")
atac = ad.read_h5ad('../h5ad/pb-atac.h5ad')
combined = ad.read_h5ad('../h5ad/pb-combined.h5ad')

/home/artem/miniconda3/envs/singlecell2/lib/python3.12/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [4]:
glue = scglue.models.load_model("glue_cat2reps_atac_120.dill")

[INFO] autodevice: Using GPU 0 as computation device.


In [25]:
atac.obsm["X_glue"] = glue.encode_data("atac", atac)
combined.obsm["X_glue"] = glue.encode_data("cttag", combined)

In [26]:
feature_embeddings = glue.encode_graph(guidance)
feature_embeddings = pd.DataFrame(feature_embeddings, index=glue.vertices)

combined.varm["X_glue"] = feature_embeddings.reindex(combined.var['id']).to_numpy()
atac.varm["X_glue"] = feature_embeddings.reindex(atac.var['id']).to_numpy()

In [43]:
# Save full datasets
atac.write_h5ad("../h5ad/glue_emb-atac-FULL.h5ad")
combined.write_h5ad("../h5ad/glue_emb-cttag-FULL.h5ad")

In [27]:
import scipy.sparse as sps
# delete matrix for smaller file size
atac.X = sps.csr_matrix(atac.X.shape)
combined.X = sps.csr_matrix(combined.X.shape)

atac.write_h5ad('../h5ad/glue_emb-atac-250217-SHORT.h5ad')
combined.write_h5ad('../h5ad/glue_emb-cttag-250217-SHORT.h5ad')